# BRFSS 2024 — Data Prep (Steps 1–5) — Google Colab

This notebook is the Colab version of `BRFSS_Recode.py`. It runs the five prep steps from the PI:

1. Count unique values in each variable
2. List what those unique values are
3. Check for remaining NaN; drop any
4. Create binary (0/1) variables by recoding (codebook labels)
5. Collapse multi-category variables into binary where sensible

**Input:** the cleaned dataset produced in PyCharm (`BRFSS_2024_cleaned.csv`, ~25 MB, or the `.parquet`, ~2 MB).

### Getting the data into Colab — pick ONE option in the next cell
- **Option A (quickest):** upload the CSV straight from your computer.
- **Option B (recommended if you'll iterate):** put the file in Google Drive and mount it — no re-uploading each session.

In [ ]:
# ===== OPTION A — upload from your computer =====
# Runs a file picker; choose BRFSS_2024_cleaned.csv (or .parquet)
from google.colab import files
uploaded = files.upload()
PATH = next(iter(uploaded))          # name of the file you picked
print('Using:', PATH)

# ===== OPTION B — mount Google Drive instead (comment out Option A) =====
# from google.colab import drive
# drive.mount('/content/drive')
# PATH = '/content/drive/MyDrive/BRFSS/BRFSS_2024_cleaned.csv'  # <- adjust path


In [ ]:
import pandas as pd

if PATH.endswith('.parquet'):
    df = pd.read_parquet(PATH)
else:
    df = pd.read_csv(PATH)

print(f'Loaded : {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

In [ ]:
VAR_LABELS = {
    'MENTHLTH':'Poor mental health days (0-30)', 'PHYSHLTH':'Poor physical health days (0-30)',
    'EXERANY2':'Any physical activity (1=Yes,2=No)', '_TOTINDA':'Meets activity guidelines (1=Yes,2=No)',
    '_SMOKER3':'Smoker (1=everyday,2=someday,3=former,4=never)', '_DRNKWK3':'Drinks per week',
    '_BMI5':'BMI x100', '_BMI5CAT':'BMI category (1=under,2=normal,3=over,4=obese)',
    '_AGE80':'Age (18-80)', 'SEXVAR':'Sex (1=Male,2=Female)', 'EDUCA':'Education (1..6, 6=college grad)',
    'INCOME3':'Income (1=<$10k .. 11=$200k+)',
    'EMPLOY1':'Employment (1=wages,2=self,3-4=out of work,5=homemaker,6=student,7=retired,8=unable)',
    '_HLTHPL2':'Health insurance (1=Have,2=None)', 'PERSDOC3':'Personal doctor (1=one,2=>one,3=No)',
    'CHECKUP1':'Last checkup (1=<1yr,2=<2yr,3=<5yr,4=5yr+,8=never)',
    'mental_risk':'TARGET: mental health risk (MENTHLTH>=14)',
}

## Step 1 — number of unique values per variable

In [ ]:
nuniq = df.nunique().sort_values()
for col, k in nuniq.items():
    print(f'{col:<12} {k:>6} unique   | {VAR_LABELS.get(col, "")}')

## Step 2 — what those unique values are

In [ ]:
for col in df.columns:
    vals = sorted(df[col].dropna().unique().tolist())
    if len(vals) <= 12:
        print(f'{col:<12} {vals}')
    else:
        print(f'{col:<12} min={min(vals):g}  max={max(vals):g}  ({len(vals)} distinct, continuous)')

## Step 3 — remaining missing (NaN) values; drop any

In [ ]:
na = df.isna().sum()
na = na[na > 0]
if na.empty:
    print('No missing values found -- nothing to drop.')
else:
    print(na)
    before = len(df)
    df = df.dropna()
    print(f'Dropped {before - len(df):,} rows -> {len(df):,} remain.')

## Steps 4 & 5 — create binary (0/1) variables

Each new column is 1 for the codebook codes listed, 0 otherwise. Multi-category variables (smoker, obese, doctor, checkup, education, employment, income) are collapsed to a single meaningful binary.

In [ ]:
BINARY_RECODES = {
    'exercise':          ('EXERANY2', {1},          'did any physical activity'),
    'meets_activity':    ('_TOTINDA', {1},          'meets activity guidelines'),
    'current_smoker':    ('_SMOKER3', {1, 2},       'smokes every/some days'),
    'obese':             ('_BMI5CAT', {4},          'BMI category = obese'),
    'female':            ('SEXVAR',   {2},          'female'),
    'insured':           ('_HLTHPL2', {1},          'has health insurance'),
    'has_pers_doctor':   ('PERSDOC3', {1, 2},       'has >=1 personal doctor'),
    'checkup_within_1yr':('CHECKUP1', {1},          'checkup within past year'),
    'college_grad':      ('EDUCA',    {6},          'college graduate (4+ yrs)'),
    'employed':          ('EMPLOY1',  {1, 2},       'employed for wages / self'),
    'low_income':        ('INCOME3',  {1, 2, 3, 4}, 'household income < $25,000'),
}

for new_col, (src, ones, meaning) in BINARY_RECODES.items():
    df[new_col] = df[src].isin(ones).astype(int)
    n1 = int(df[new_col].sum()); pct = n1 / len(df) * 100
    print(f'{new_col:<20} <- {src:<9} ({meaning}): 1={n1:,} ({pct:.1f}%)')

In [ ]:
# Verify one recode (change the source name to spot-check others)
pd.crosstab(df['_SMOKER3'], df['current_smoker'])

## Save / download the recoded dataset

In [ ]:
df.to_csv('BRFSS_2024_recoded.csv', index=False)
print('Saved BRFSS_2024_recoded.csv  ', df.shape)

# To download it back to your computer:
from google.colab import files
files.download('BRFSS_2024_recoded.csv')

# Or, if you mounted Drive (Option B), save straight to Drive instead:
# df.to_csv('/content/drive/MyDrive/BRFSS/BRFSS_2024_recoded.csv', index=False)